Testing draft pick and then value from draft value chart as baselines/comparisons to see how predictive they are of draft AV. Goal is to beat that.

In [74]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline


df = pd.read_csv("draft_values_vs_av_processed.csv")

df = df[['pick', 'dr_av']].dropna()

X_pick = df[['pick']]
y = df['dr_av']

# linear regression
lr = LinearRegression()
scores_pick = cross_val_score(lr, X_pick, y, cv=5, scoring='r2')
print("Linear Regression (pick) CV R²:", scores_pick.mean())

# linear regression on log(pick)
df['log_pick'] = np.log(df['pick'])
X_log = df[['log_pick']]

scores_log = cross_val_score(lr, X_log, y, cv=5, scoring='r2')
print("Linear Regression (log pick) CV R²:", scores_log.mean())

Linear Regression (pick) CV R²: -0.34245537094091
Linear Regression (log pick) CV R²: 0.02215577757717193


In [75]:
df = pd.read_csv("draft_values_vs_av_processed.csv")

cols = ['stuart', 'johnson', 'hill', 'otc', 'pff', 'dr_av']
df = df[cols].dropna()

X_values = df[['stuart', 'johnson', 'hill', 'otc', 'pff']]
y = df['dr_av']

# linear regression
scores_all = cross_val_score(lr, X_values, y, cv=5, scoring='r2')
print("Linear Regression (all value charts) CV R²:", scores_all.mean())

# model for each value chart using linear regression
for col in ['stuart', 'johnson', 'hill', 'otc', 'pff']:
    X_single = df[[col]]
    score = cross_val_score(lr, X_single, y, cv=5, scoring='r2').mean()
    print(f"Linear Regression using {col} only CV R²:", score)

Linear Regression (all value charts) CV R²: -0.2989119515549348
Linear Regression using stuart only CV R²: 0.02240141950861807
Linear Regression using johnson only CV R²: -0.1961243470333538
Linear Regression using hill only CV R²: -0.20044901161519957
Linear Regression using otc only CV R²: 0.02216579584092624
Linear Regression using pff only CV R²: 0.011054168738439719


In [77]:
df = df[['pff', 'dr_av']].dropna()

X = df[['pff']]
y = df['dr_av']

# degree 2-4 polynomial
for d in [2,3,4]:
    model = Pipeline([("poly", PolynomialFeatures(degree=d)), ("lr", LinearRegression())])
    score = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
    print(f"Degree {d} polynomial CV R²:", score)

Degree 2 polynomial CV R²: -0.012329936873904712
Degree 3 polynomial CV R²: -2.101736309658238
Degree 4 polynomial CV R²: -44.040758675509515


Draft value charts and pick number are very weak predictors of draft approximate 